# CHILI Case 2: Venus

In [14]:
import sys
sys.path.append('/home/maxime/Models/MOAI/src')

from physics import eos,viscosity
from physics.phase_change.refractories import mc_book
from chemistry.elements import *
from chemistry.molecules import *
from chemistry.equilibria import *
from chemistry.redox import *
from magma_ocean import magma_ocean # fractional_crystallization as magma_ocean
from atmospheres import multi_species_single_condensible_convective_atmosphere as atmosphere
from physics.eos import eos_book
from utils import y2s
import matplotlib.pyplot as plt

import exo_k as xk
xk.Settings().set_mks(True)
xk_datapath = '/home/maxime/Models/Radiative_Transfer/data/'

%matplotlib inline

eq_C.coefs[O2]      = 0
eq_H.coefs[O2]      = 0

## Inputs

In [15]:
Rp          = 0.95*6.3781e6 # planetary radius             [m]  from CHILI Protocol paper
Rc          = 0.55*Rp  # core radius                  [m]  from CHILI Protocol paper
g           = 9.81*(0.815/0.95**2)     # surface gravity              [m/s^2]
T0          = 3000.    # initialpotential temperature [K]
p_CMB       = 135e9    # CMB pressure                 [Pa]

semimajor_axis      = 0.723 # [AU]
Bond_albedo         = 0.1   # [-]     from CHILI Protocol paper
flux1AU             = 920. # [W/m^2] from CHILI Protocol paper
stellar_irradiation = 1./4*flux1AU/(semimajor_axis**2)*(1.-Bond_albedo)

## Model

### Thermal evolution model

In [19]:
# create MO
MO = magma_ocean(T0,
                 p_CMB,
                 #eos={'rho':eos_book['rho']['MK19l'],'alpha':eos_book['alpha']['N19'],'cp':eos_book['cp']['cst']},
                 g=g,
                 R=Rp,
                 ConvCum=True)

# add melting curves
from physics.phase_change.refractories import mc_book
MO.setMeltingCurves(mc_book['Earth'],RCMF=0.4)
MO.updateT_pot(max(MO.T_pot_lookup))

### Redox model

In [19]:
# Set Fe3*/FeT in MO
from chemistry.partition_coefficients import get_part_coef
Fe_ferrous.part_coef = 1
MO.setParametrization('D_Fe3+_profile',lambda var: get_part_coef(var['pressures'],'Fe3+','2000km'),['pressures'],ptype='profile',channel='')

def set_Fe_ferric_part_coef(MO):
    # integrate the D_Fe3+ profile over
    Fe_ferric.part_coef = MO.getAverage(MO.profiles['D_Fe3+_profile'],domain=[MO.p_bot,MO.p_liq],mass_weighted=True)
    if np.isnan(Fe_ferric.part_coef):
        Fe_ferric.part_coef = get_part_coef(MO.p_bot,'Fe3+','2000km')
    return Fe_ferric.part_coef
MO.setParametrization('D_Fe3+_eff', lambda var: set_Fe_ferric_part_coef(var['MO']),[],ptype='scalar')

BSE_Fe3p_FeT = 0.094 # 0.17 # 0.094 is tuned to have IW-4 when the potential temperature is 1600 K (8.7% MO remaining). 0.17 is tuned to have IW-4 at 1600 K when the MO is 660 km-de8
BSE_FeO      = 0.0583 # Sun and McDonnough 1995
MO.addSpecies([Fe_ferric,Fe_ferrous],
              [BSE_Fe3p_FeT*BSE_FeO,(1-BSE_Fe3p_FeT)*BSE_FeO])
MO.to_frac=['Fe3+','Fe2+']
MO.fractionation(MO.to_frac)
MO.setParametrization('Fe3+/FeT',lambda var:1/(1+1/(var['Fe3+_liquid']/var['Fe2+_liquid']*Fe_ferrous.molecular_mass/Fe_ferric.molecular_mass)),
                      ['Fe2+_liquid','Fe3+_liquid'],ptype='scalar')

# Set fO2
from physics.constants import BSE
MO.setParametrization('fO2_sfc',
                      lambda var: fO2_sfc_H22(var['T_pot'],var['Fe3+/FeT'],BSE),
                      ['T_pot','Fe3+/FeT'],
                      ptype='scalar')
from chemistry.redox import fO2_buffers_H22
MO.setParametrization('DeltaIW_sfc',
                      lambda var: np.log10(var['fO2_sfc'])-fO2_buffers_H22(var['p_sfc'],var['T_pot']),
                      ['T_pot','p_sfc','fO2_sfc'],
                      ptype='scalar')

### Chemical model

In [19]:
M_H2O = 4.7e21 # [kg]
M_CO2 = 1e21   # [kg]

In [19]:
MO.setupChemistry({H2O:M_ini['H2O']/MO.M_system,
                   CO2:M_ini['CO2']/MO.M_system,
                   N2:1e-6,
                   S2:1e-6})
MO.speciationInit()

### Atmosphere

In [19]:
MO.atm = atmosphere(list(MO.atm.species.values()),
                    np.array([MO.atm.partial_pressure[sp.formula] for sp in list(MO.atm.species.values())]),
                    MO.adiabat.T_pot,R=MO.R_out,g=MO.gravity,pToA=1,cond=H2O,T_str=200.)

#### Use exo_k for radiative transfer        

In [19]:
xk.Settings().add_search_path(xk_datapath + 'corrk',path_type='ktable')
corrk_database = xk.Kdatabase(['H2O','CO2','CO','NH3','CH4','H2S','HCN','SO2'])

R=10
wavenumber_grid=xk.wavenumber_grid_R(200.,10000.,R)
corrk_database.bin_down(wavenumber_grid)

xk.Settings().add_search_path(xk_datapath + 'cia',path_type='cia')
cia_database = xk.CIAdatabase(molecules=['H2'])
cia_database.sample(corrk_database.wns)

R = sum([8.314/(molecules_book[sp].molecular_mass/1000)*MO.atm.volume_mixing_ratio[sp] for sp in MO.atm.species])
cp = sum([molecules_book[sp].cp_mass(MO.atm.Ts)*MO.atm.volume_mixing_ratio[sp] for sp in MO.atm.species])
rcp = R/cp

logplay = np.log10(np.flipud(MO.atm.profiles['pressure']))
tlay    = np.flipud(MO.atm.profiles['temperature'])
clay    = {sp:np.flipud(MO.atm.profiles['molar fraction'][sp]) for sp in MO.atm.species}

MO.atm.radiation = xk.Atm(logplay=logplay,
                          tlay=tlay,
                          grav=MO.gravity, Rp=MO.R_out, rcp=rcp,
                          albedo_surf=Bond_albedo,
                          composition=clay,
                          k_database=corrk_database,
                          cia_database=cia_database,
                          rayleigh=True)
def getOLR():
    return MO.atm.radiation.emission_spectrum_2stream().total

MO.atm.getOLR = getOLR

def update_rad_pres(var):
    MO.atm.radiation.set_logPT_profile(np.log10(np.flipud(var['atm'].profiles['pressure'])),np.flipud(var['atm'].profiles['temperature']))
MO.setParametrization('radPres',update_rad_pres,['atm'],is_profile=False,channel='post-spec',ptype='action')  

In [19]:
def flux_residual(MO,T):
    #print('Calculating flux residual for T_surf=',T)
    MO.atm.updateTs(T)
    MO.updateBL(MO.adiabat.T_pot-T)
    # now need to update composition (can be altered by condensation) and tempreature profiles
    MO.atm.radiation.set_gas({sp:np.flipud(MO.atm.profiles['molar fraction'][sp]) for sp in MO.atm.species})
    MO.atm.radiation.set_T_profile(np.flipud(MO.atm.profiles['temperature']))
    return MO.BL.getFlux()+stellar_irradiation-MO.atm.getOLR()
MO.flux_residual = flux_residual

MO.atm.stellar_irradiation = stellar_irradiation
MO.RK4_step(0)

## Outputs

In [19]:
from tools import time_series
ts=time_series(MO)

# Add custom quantities
ts.register('OLR','atm.getOLR()')                          # Save OLR
ts.register('mu_atm','atm.average_molecular_mass')         # Save average molecular mass of the atmosphere
ts.register('p_tot','atm.ps')                              # Save surface pressure
ts.register('p_cloud','atm.cloud_deck')                    # Save lifting condensation level
ts.register('T_bot','adiabat.getT(self.attribute.p_bot)')  # Save temperature at the bottom of the MO
ts.register('Ra','getRa(\'MO\')')                          # Save MO Rayleigh number
ts.register('phi_H2','scalar[\'phi_H2\']')                 # Save H2 escape flux
ts.register('Fconv','BL.getFlux()')                        # Save MO convective flux
ts.register('T_surf','atm.Ts')                             # Save surface temperature

ts.write(0)

### Timestepping

In [19]:
MO.max_dT=100000*y2s
dt = 10*y2s
#dt /= 2

MO.max_change['Tpot']= 5
MO.min_T_surf = 1000

### Run

In [19]:
while MO.M_system/ts('M_sys')[0] > 0.01:
    dt=MO.RK4_step(dt,rtol=0.001)
    ts.write(MO.t)

In [19]:
M_atm = 4*np.pi*MO.R_out**2/MO.gravity*sum([ts('p_'+str(sp)) for sp in MO.atm.species])

M_H_atm = M_atm*(ts('p_H2O')/ts('p_tot')*H2O.molecular_mass/ts('mu_atm')*2*H.atomic_mass/H2O.molecular_mass \
               + ts('p_H2')/ts('p_tot')*H2.molecular_mass/ts('mu_atm')*2*H.atomic_mass/H2.molecular_mass \
               + ts('p_CH4')/ts('p_tot')*CH4.molecular_mass/ts('mu_atm')*4*H.atomic_mass/CH4.molecular_mass)
M_C_atm = M_atm*(ts('p_CO2')/ts('p_tot')*CO2.molecular_mass/ts('mu_atm')*C.atomic_mass/CO2.molecular_mass \
               + ts('p_CO')/ts('p_tot')*CO.molecular_mass/ts('mu_atm')*C.atomic_mass/CO.molecular_mass \
               + ts('p_CH4')/ts('p_tot')*CH4.molecular_mass/ts('mu_atm')*C.atomic_mass/CH4.molecular_mass)

import pandas as pd
ts_pd = pd.DataFrame({'t(yr)':ts('t')/y2s,
                      'T_surf(K)':ts('T_pot'),
                      'T_pot(K)':ts('T_sfc'),
                      'flux_surf(W/m2)':ts('Fconv'),
                      'flux_OLR(W/m2)':ts('OLR'),
                      'flux_ASR(W/m2)':np.ones_like(ts('t'))*stellar_irradiation,
                      'phi(vol_frac)':ts('phi_avg'),
                     #'fO2_solid(bar)':[],
                      'fO2_melt(bar)':ts('fO2_sfc'),
                     #'thick_surf_bl(m)':[],
                      'massC_solid(kg)':ts('C_solid')*ts('M_sol'),
                      'massC_melt(kg)':ts('C_liquid')*ts('M_liq'),
                      'massC_atm(kg)':M_H_atm,
                      'massH_solid(kg)':ts('H_solid')*ts('M_sol'),
                      'massH_melt(kg)':ts('H_liquid')*ts('M_liq'),
                      'massH_atm(kg)':M_C_atm,
                      'p_surf(bar)':ts('p_tot')*1e-5,
                      'p_H2O(bar)':ts('p_H2O')*1e-5,
                      'p_CO2(bar)':ts('p_CO2')*1e-5,
                      'p_CO(bar)':ts('p_CO')*1e-5,
                      'p_H2(bar)':ts('p_H2')*1e-5,
                      'p_CH4(bar)':ts('p_CH4')*1e-5,
                      'p_O2(bar)':ts('fO2_sfc'),
                      'mmw(kg/mol)':ts('mu_atm')*1e-3,
                      'R_solid(m)':np.ones_like(ts('t')*MO.R_out)})
ts_pd.to_csv(f'evolution-moai-earth-grid-{M_H2O}H-{M_CO2}C.csv',index=None)

No file found: calculating lookups, be patient it can take some time!
No file found: calculating lookups, be patient it can take some time!
No file found: calculating lookups, be patient it can take some time!
No file found: calculating lookups, be patient it can take some time!
No file found: calculating lookups, be patient it can take some time!
No file found: calculating lookups, be patient it can take some time!
No file found: calculating lookups, be patient it can take some time!
No file found: calculating lookups, be patient it can take some time!
No file found: calculating lookups, be patient it can take some time!
